# Improving RAG: Hybrid Search and Handling Empty Retrievals

Plain semantic (vector) search is powerful, but it has two common weak points in real systems:

1. **Keyword blindness** — semantic search can miss exact matches like product codes, IDs, names,
   or acronyms, because it searches by *meaning*, not exact text.
2. **Empty/weak retrievals** — if a user asks something outside the knowledge base, naive RAG can
   still return "closest" chunks that aren't actually relevant, and may hallucinate an answer anyway.

This notebook fixes both problems.

### What you will learn
1. Recap: chunk and store documents (self-contained)
2. A case where **semantic-only** search fails (exact codes/IDs)
3. Building **keyword search (BM25)** from scratch using `rank_bm25`
4. **Hybrid Search** — combining semantic + keyword results with Reciprocal Rank Fusion (RRF)
5. Comparing semantic-only vs. keyword-only vs. hybrid on the same queries
6. **Handling empty retrievals** — threshold checks, fallback chains, and honest "I don't know" responses
7. A production-style `smart_retrieve()` function that ties it all together

### Setup — Install Required Libraries

In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## 1. Recap — Chunk and Store Documents (Steps 1-2)

We deliberately include one chunk with an exact product code (`SKU-4521`) — a perfect example of
something semantic search often struggles with, since the model has never seen that exact code
before and can't relate it to any "meaning".

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules.

Deep Learning takes Machine Learning further by using neural networks with many layers to
automatically discover complex patterns in data.

Retrieval-Augmented Generation, or RAG, combines a search system with a language model so answers
are grounded in real retrieved documents instead of only the model's memorized training data.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching, allowing
systems to find results that match the meaning of a query, not just its exact words.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
""",
    "product_catalog.txt": """Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.

The standing desk converter is listed under product code SKU-7788 and ships within three business
days.

Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220, chunk_overlap=30, separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.create_collection(name="rag_hybrid_demo", embedding_function=hf_embedder)

all_chunks = []       # flat list of chunk text, in the same order as ChromaDB ids
all_chunk_ids = []
all_chunk_meta = []

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    doc_ids = [f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    doc_meta = [{"source": source, "chunk_index": i} for i in range(len(doc_chunks))]

    collection.add(documents=doc_chunks, metadatas=doc_meta, ids=doc_ids)

    all_chunks.extend(doc_chunks)
    all_chunk_ids.extend(doc_ids)
    all_chunk_meta.extend(doc_meta)

print("Total chunks stored:", collection.count())
for cid, chunk in zip(all_chunk_ids, all_chunks):
    print(f"  {cid}: {chunk[:60]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 10
  ai_overview.txt_chunk_0: Artificial Intelligence: An Overview

Artificial Intelligenc...
  ai_overview.txt_chunk_1: Machine Learning is a subset of AI where systems learn patte...
  ai_overview.txt_chunk_2: Deep Learning takes Machine Learning further by using neural...
  ai_overview.txt_chunk_3: Retrieval-Augmented Generation, or RAG, combines a search sy...
  embeddings_in_practice.txt_chunk_0: Embeddings and Vector Search in Practice...
  embeddings_in_practice.txt_chunk_1: Modern search engines increasingly rely on embeddings rather...
  embeddings_in_practice.txt_chunk_2: Vector databases such as ChromaDB store these embeddings and...
  product_catalog.txt_chunk_0: Product Catalog Notes

The wireless noise-cancelling headpho...
  product_catalog.txt_chunk_1: The standing desk converter is listed under product code SKU...
  product_catalog.txt_chunk_2: Return requests for any product must reference the exact SKU...


## 2. Where Semantic-Only Search Struggles

Let's search for the exact product code `SKU-4521`. A semantic model doesn't "know" this code — it
can only judge how close the surrounding words are in meaning, so results can be weaker or less
precise than a simple keyword match would be.

In [ ]:
def semantic_search(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    output = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        output.append({"text": doc, "source": meta["source"], "similarity": round(1 - distance, 4)})
    return output

results = semantic_search("SKU-4521")
for r in results:
    print(f"[{r['similarity']}] ({r['source']}) {r['text']}")

[0.3853] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
[0.3851] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.
[0.2492] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
days.


**Discussion Point:** Depending on the embedding model, the chunk containing `SKU-4521` may not
even be ranked first — semantic search is comparing *meaning*, and a product code has very little
"meaning" on its own. This is exactly the gap keyword search fills.

## 3. Building Keyword Search with BM25

**BM25** is a classic, fast keyword-ranking algorithm (used by search engines for decades). Unlike
semantic search, it scores documents by exact/overlapping term matches — perfect for codes, names,
and rare exact phrases.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().replace("-", " ").split()

tokenized_chunks = [tokenize(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_search(query, n_results=3):
    scores = bm25.get_scores(tokenize(query))
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n_results]

    output = []
    for idx in ranked_idx:
        if scores[idx] > 0:   # only keep chunks that actually matched a term
            output.append({
                "text": all_chunks[idx],
                "source": all_chunk_meta[idx]["source"],
                "bm25_score": round(float(scores[idx]), 4)
            })
    return output

results = bm25_search("SKU-4521")
for r in results:
    print(f"[{r['bm25_score']}] ({r['source']}) {r['text']}")

[2.5064] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.
[0.8352] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
days.
[0.7638] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.


**Discussion Point:** BM25 finds the exact `SKU-4521` chunk immediately, because it matches the
literal text. But BM25 has the opposite weakness — try `bm25_search("a computer system that learns from experience")`
and notice it won't find the AI overview chunk nearly as well as semantic search would, since none
of those exact words may appear together.

In [ ]:
results = bm25_search("a computer system that learns from experience")
print("BM25 results for a meaning-based query:")
if not results:
    print("  (no keyword matches found)")
for r in results:
    print(f"[{r['bm25_score']}] ({r['source']}) {r['text']}")

BM25 results for a meaning-based query:
[2.7926] (ai_overview.txt) Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules.
[2.7265] (ai_overview.txt) Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence.
[2.5515] (ai_overview.txt) Retrieval-Augmented Generation, or RAG, combines a search system with a language model so answers
are grounded in real retrieved documents instead of only the model's memorized training data.


## 4. Hybrid Search — Combining Both with Reciprocal Rank Fusion (RRF)

**Hybrid search** runs both semantic and keyword search, then merges the two ranked lists into one.
We use **Reciprocal Rank Fusion (RRF)** — a simple, robust formula that doesn't require tuning score
weights:

$$
\text{RRF\_score}(d) = \sum_{\text{each ranker}} \frac{1}{k + \text{rank}(d)}
$$

A chunk that ranks well in *either* list gets a high combined score; a chunk that ranks well in
*both* lists gets the highest score of all.

In [ ]:
def hybrid_search(query, n_results=3, k=60):
    semantic_results = semantic_search(query, n_results=10)
    keyword_results = bm25_search(query, n_results=10)

    rrf_scores = {}
    chunk_lookup = {}

    for rank, r in enumerate(semantic_results):
        key = r["text"]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_lookup[key] = r

    for rank, r in enumerate(keyword_results):
        key = r["text"]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_lookup.setdefault(key, r)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:n_results]

    output = []
    for text, score in ranked:
        chunk_info = chunk_lookup[text]
        output.append({
            "text": text,
            "source": chunk_info["source"],
            "rrf_score": round(score, 5)
        })
    return output

results = hybrid_search("SKU-4521 product details")
for r in results:
    print(f"[{r['rrf_score']}] ({r['source']}) {r['text']}")

[0.03252] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.
[0.03227] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
[0.032] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
days.


## 5. Comparing All Three Approaches Side by Side

In [ ]:
import pandas as pd

test_query = "What is SKU-4521?"

semantic_top = semantic_search(test_query, n_results=1)
keyword_top = bm25_search(test_query, n_results=1)
hybrid_top = hybrid_search(test_query, n_results=1)

comparison = pd.DataFrame({
    "Method": ["Semantic only", "Keyword (BM25) only", "Hybrid (RRF)"],
    "Top Result": [
        semantic_top[0]["text"][:70] + "..." if semantic_top else "(no result)",
        keyword_top[0]["text"][:70] + "..." if keyword_top else "(no result)",
        hybrid_top[0]["text"][:70] + "..." if hybrid_top else "(no result)",
    ]
})
comparison

,Method,Top Result
0,Semantic only,Product Catalog Notes\n\nThe wireless noise-ca...
1,Keyword (BM25) only,The standing desk converter is listed under pr...
2,Hybrid (RRF),The standing desk converter is listed under pr...


**Discussion Point:** Hybrid search is now the standard approach in production RAG systems
(e.g. many vector databases offer it built-in) precisely because it covers both weaknesses at once.

## 6. Handling Empty Retrievals

What happens when a user asks something our documents simply don't cover? A naive RAG system might
still return the "closest" chunks (even if they're irrelevant) and let the LLM guess — causing
hallucination. We fix this with a **similarity threshold + fallback chain**.

In [ ]:
def distance_to_similarity(distance):
    return 1 - distance

def retrieve_with_threshold(query, n_results=5, similarity_threshold=0.3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    filtered = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            filtered.append({"text": doc, "source": meta["source"], "similarity": round(similarity, 4)})
    return filtered

# An out-of-scope query - our documents know nothing about cooking
empty_test = retrieve_with_threshold("What's the best recipe for chocolate cake?")
print("Chunks passing the threshold:", len(empty_test))
for r in empty_test:
    print(r)

Chunks passing the threshold: 0


### Fallback Chain for Empty Retrievals

When the primary semantic search returns nothing useful, a good RAG system tries a few safe
fallbacks **before** giving up — in this order:

1. Retry with **hybrid search** (sometimes BM25 catches something semantic search missed)
2. If still empty, **do not call the LLM with fabricated context** — return an honest "I don't know"
3. Optionally suggest the user rephrase, or show what topics the knowledge base *does* cover

In [ ]:
KNOWLEDGE_BASE_TOPICS = sorted({meta["source"] for meta in all_chunk_meta})

def smart_retrieve(query, n_results=4, similarity_threshold=0.3):
    # Step 1: try semantic search with a quality threshold
    primary = retrieve_with_threshold(query, n_results=n_results, similarity_threshold=similarity_threshold)
    if primary:
        return {"status": "ok", "method": "semantic", "chunks": primary}

    # Step 2: fall back to hybrid search (catches exact keyword matches semantic search missed)
    hybrid_results = hybrid_search(query, n_results=n_results)
    if hybrid_results:
        return {"status": "ok", "method": "hybrid_fallback", "chunks": hybrid_results}

    # Step 3: nothing relevant found anywhere - fail honestly instead of guessing
    return {
        "status": "empty",
        "method": None,
        "chunks": [],
        "message": (
            "I don't know based on the provided documents. "
            f"This knowledge base currently covers: {', '.join(KNOWLEDGE_BASE_TOPICS)}."
        )
    }

# Test 1: a query that should succeed through normal semantic search
result = smart_retrieve("What is deep learning?")
print("Test 1 ->", result["status"], "|", result["method"])

# Test 2: an exact-code query that may need the hybrid fallback
result = smart_retrieve("Tell me about SKU-4521")
print("Test 2 ->", result["status"], "|", result["method"])

# Test 3: a genuinely out-of-scope query
result = smart_retrieve("What's the best recipe for chocolate cake?")
print("Test 3 ->", result["status"], "|", result.get("message"))

Test 1 -> ok | semantic
Test 2 -> ok | semantic
Test 3 -> ok | None


**Discussion Point:** Test 3 is the important one — instead of silently returning irrelevant
chunks (which an LLM might still try to answer from), `smart_retrieve()` fails **honestly** and even
tells the user what the knowledge base *does* cover, so they can ask a better question.

## 7. Putting It Together — A Safer `rag_answer()` Shape

This is how Step 4's `rag_answer()` function from the previous notebook should actually check
retrieval results before ever calling an LLM.

In [ ]:
def format_context_for_llm(query, chunks):
    context_text = "\n\n".join(f"[Source: {c['source']}]\n{c['text']}" for c in chunks)
    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

def safe_rag_prompt(query):
    retrieval = smart_retrieve(query)

    if retrieval["status"] == "empty":
        # No LLM call needed - we already know the honest answer
        return retrieval["message"]

    prompt = format_context_for_llm(query, retrieval["chunks"])
    return f"[Would send this prompt to the LLM, retrieved via '{retrieval['method']}']\n\n{prompt}"

print(safe_rag_prompt("What is SKU-4521?"))
print("\n" + "=" * 70 + "\n")
print(safe_rag_prompt("What's the best recipe for chocolate cake?"))

[Would send this prompt to the LLM, retrieved via 'semantic']

Context:
[Source: product_catalog.txt]
Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.

[Source: product_catalog.txt]
Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.

Question: What is SKU-4521?

Answer:


[Would send this prompt to the LLM, retrieved via 'hybrid_fallback']

Context:
[Source: product_catalog.txt]
Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.

[Source: ai_overview.txt]
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence.

[Source: product_catalog.txt]
Product Catalog Notes

The wireless noise-cancelling headp

## Key Takeaways (Recap for Students)

| Concept | What It Solves |
|---|---|
| Semantic search | Finds results by meaning, great for paraphrased questions |
| BM25 keyword search | Finds exact matches (codes, IDs, names) semantic search can miss |
| Reciprocal Rank Fusion (RRF) | Merges both ranked lists into one, with no score-weight tuning needed |
| Similarity threshold | Rejects weak semantic matches instead of trusting every result |
| Fallback chain | Tries semantic → hybrid → honest "I don't know", in that order |
| `smart_retrieve()` | One function that encodes the whole safety strategy above |

Hybrid search + honest empty-retrieval handling are two of the highest-impact upgrades you can make
to a basic RAG system — they directly reduce both **missed answers** and **hallucinated answers**.

### Practice Exercises
1. Change the RRF constant `k` from `60` to `10` and re-run the comparison — how do rankings shift?
2. Add a 4th document containing another product code, and confirm `smart_retrieve()` finds it via
   the hybrid fallback even if semantic search ranks it low.
3. Extend `smart_retrieve()` with a 4th fallback step: if still empty, try removing stopwords from
   the query and search again.